# NutriVisión MX: pipeline completo en Google Colab

Este notebook corre **todo el entrenamiento y la evaluación en GPU** y al final descarga un `.zip` con los resultados para copiarlos al repositorio.

**Un solo archivo:** este notebook ya contiene todo el código.

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → **T4 GPU*** y luego *Ejecutar todo*.

Tiempo aproximado: **40–60 min** (la descarga de Food-101 son ~5 GB por red; el disco de Colab no se llena porque se lee como *stream*).

## 1. Verificar GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Extraer el código del proyecto
El código completo del repositorio viene **dentro de este notebook** (comprimido en la celda siguiente), así que no hay que subir nada más.

In [ ]:
import base64, io, os, zipfile
CODIGO = "..."
os.chdir('/content')
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODIGO))).extractall('/content')
os.chdir('/content/nutrivision-mx')
print(sorted(os.listdir('.')))

## 3. Dependencias
Colab ya trae PyTorch, OpenCV, scikit-learn y pandas; solo se instalan las que faltan. (Las versiones exactas usadas localmente están en `requirements.txt`.)

In [ ]:
!pip install -q imagehash onnx onnxruntime onnxscript
import torch, torchvision; print(torch.__version__, torchvision.__version__, torch.cuda.get_device_name(0))

## 4. Datos: descarga y particiones sin fuga

In [ ]:
!python data/download.py
!python -m src.make_splits

## 5. Entrenamiento de las dos arquitecturas (mismo protocolo)

In [ ]:
!python -m src.train --arch mobilenet_v3_large --workers 2

In [ ]:
!python -m src.train --arch efficientnet_b0 --workers 2

## 6. Evaluación en prueba (una sola vez) y fuera de distribución (ISIA Food-500)

In [ ]:
!python -m src.evaluate --arch mobilenet_v3_large --split test
!python -m src.evaluate --arch efficientnet_b0 --split test
!python -m src.download_ood
!python -m src.evaluate --arch mobilenet_v3_large --split ood

## 7. Interpretabilidad, exportación a ONNX/INT8 y comparación
La latencia aquí se mide en el **CPU de Colab**; la del equipo de la demo se mide con `app/demo.py` en la laptop.

In [ ]:
!python -m src.gradcam --arch mobilenet_v3_large
!python -m src.export --arch mobilenet_v3_large --threads 2
!python -m src.export --arch efficientnet_b0 --threads 2 --f1-subset 600 --calib-n 64
!python -m src.compare

## 8. Ejecutar los notebooks de análisis (quedan guardados con sus salidas)

In [ ]:
for nb in ["notebooks/01_exploracion_datos.ipynb", "notebooks/02_curvas_entrenamiento.ipynb",
           "notebooks/03_revision_etiquetas.ipynb", "eval/robustez.ipynb",
           "eval/auditoria_subgrupos.ipynb", "eval/analisis_errores.ipynb"]:
    print("==>", nb)
    !jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=3600 "$nb" 2>&1 | grep -E "Writing|Error" 

## 9. Video de entrada para la demo

In [ ]:
!python app/make_demo_video.py --out app/demo.mp4

## 10. Empaquetar y descargar resultados
Descomprime este zip **dentro de la carpeta del proyecto** en tu computadora (sobrescribe `runs/`, `docs/`, `notebooks/`, `eval/` y agrega `app/demo.mp4`).

In [ ]:
from google.colab import files
!rm -f /content/resultados.zip
!zip -qr /content/resultados.zip runs docs notebooks eval app/demo.mp4 data/splits/reporte_particiones.json data/splits/removidas_por_fuga.csv data/ood/labels.csv -x "runs/*/export/model_pre.onnx"
!du -h /content/resultados.zip
files.download("/content/resultados.zip")

## 11. Demo en vivo con la webcam de tu laptop  <!-- WEBCAM_NUTRIVISION -->
Esta parte va **después** de descargar los resultados, para que "Ejecutar todo" no se quede esperando la cámara.

1. Ejecuta la celda de **preparación** (carga el modelo entrenado en la GPU).
2. Ejecuta la celda de **video en vivo**. El navegador te pedirá permiso para la cámara: acéptalo.
3. Muestra la comida (o una foto en tu celular) frente a la cámara. Se ve el platillo, la confianza, las calorías aproximadas, el top-3 y los FPS.
4. Mueve el deslizador para cambiar el **umbral de confianza**. Por debajo del umbral aparece *"No estoy seguro"*.
5. Presiona **Detener** para terminar.

> Colab corre en un servidor, así que cada cuadro viaja de tu navegador a Google y de regreso. Los FPS dependen de tu internet (normalmente 5–15).
> La demo oficial de ≥10 FPS es la local: `python app/demo.py --source 0`.

In [ ]:
# WEBCAM_NUTRIVISION: preparación
import base64, io, json, time
import numpy as np, torch
from PIL import Image
from IPython.display import display, Javascript
from google.colab.output import eval_js
from src.config import CLASSES, DISPLAY, KCAL, OTHER_CLASS
from src.data import get_transforms
from src.evaluate import load_model

DEV = "cuda" if torch.cuda.is_available() else "cpu"
MODELO = load_model("mobilenet_v3_large", DEV)
TF = get_transforms(train=False)

@torch.no_grad()
def clasificar(img: Image.Image) -> np.ndarray:
    x = TF(img.convert("RGB")).unsqueeze(0).to(DEV)
    return torch.softmax(MODELO(x), 1)[0].cpu().numpy()

JS_WEBCAM = """
var nvDiv=null, nvVideo=null, nvCanvas=null, nvLabel=null, nvSlider=null, nvStop=false;
async function nvCreate(){
  nvDiv=document.createElement('div');
  nvDiv.style.cssText='position:relative;width:640px;font-family:sans-serif';
  nvVideo=document.createElement('video');
  nvVideo.autoplay=true; nvVideo.playsInline=true; nvVideo.muted=true;
  nvVideo.style.cssText='width:640px;border-radius:8px;background:#000';
  nvLabel=document.createElement('div');
  nvLabel.style.cssText='position:absolute;top:10px;left:10px;right:10px;padding:8px 12px;'+
    'background:rgba(0,0,0,.65);color:#fff;border-radius:6px;font-size:16px;line-height:1.35';
  nvLabel.innerHTML='Iniciando cámara...';
  var ctr=document.createElement('div'); ctr.style.cssText='margin-top:8px;display:flex;gap:12px;align-items:center';
  var lab=document.createElement('span'); lab.innerHTML='Umbral: <b id="nvThr">55</b>%';
  nvSlider=document.createElement('input'); nvSlider.type='range'; nvSlider.min=0; nvSlider.max=100; nvSlider.value=55;
  nvSlider.oninput=function(){document.getElementById('nvThr').textContent=nvSlider.value;};
  var btn=document.createElement('button'); btn.textContent='Detener'; btn.onclick=function(){nvStop=true;};
  ctr.appendChild(lab); ctr.appendChild(nvSlider); ctr.appendChild(btn);
  nvDiv.appendChild(nvVideo); nvDiv.appendChild(nvLabel); nvDiv.appendChild(ctr);
  document.querySelector('#output-area').appendChild(nvDiv);
  var stream=await navigator.mediaDevices.getUserMedia({video:{width:640,height:480}});
  nvVideo.srcObject=stream; await nvVideo.play();
  nvCanvas=document.createElement('canvas'); nvCanvas.width=320; nvCanvas.height=320;
  google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
}
async function nvFrame(labelHtml){
  if(nvStop){
    nvVideo.srcObject.getVideoTracks().forEach(function(t){t.stop();});
    nvLabel.innerHTML='Demo detenida.'; nvStop=false; nvDiv=null; return '';
  }
  if(!nvDiv){ await nvCreate(); }
  if(labelHtml){ nvLabel.innerHTML=labelHtml; }
  var w=nvVideo.videoWidth, h=nvVideo.videoHeight, s=Math.min(w,h);
  nvCanvas.getContext('2d').drawImage(nvVideo,(w-s)/2,(h-s)/2,s,s,0,0,320,320);
  return JSON.stringify({img:nvCanvas.toDataURL('image/jpeg',0.8), thr:Number(nvSlider.value)/100});
}
"""
print("Modelo listo en", DEV)


In [ ]:
# WEBCAM_NUTRIVISION: video en vivo (presiona "Detener" para terminar)
display(Javascript(JS_WEBCAM))
SUAVIZADO = 0.7            # promedio móvil exponencial de las probabilidades (evita parpadeo)
ema, etiqueta, t_prev, fps, lat_ms = None, "", time.time(), 0.0, []

while True:
    resp = eval_js(f"nvFrame({json.dumps(etiqueta)})")
    if not resp:
        break
    d = json.loads(resp)
    img = Image.open(io.BytesIO(base64.b64decode(d["img"].split(",", 1)[1])))
    arr = np.asarray(img)

    t0 = time.time()
    p = clasificar(img)
    lat_ms.append((time.time() - t0) * 1000)
    ema = p if ema is None else SUAVIZADO * ema + (1 - SUAVIZADO) * p

    ahora = time.time(); fps = 0.8 * fps + 0.2 / max(ahora - t_prev, 1e-3); t_prev = ahora
    k = int(ema.argmax()); conf = float(ema[k]); clase = CLASSES[k]
    top3 = " · ".join(f"{DISPLAY[CLASSES[i]]} {ema[i]:.0%}" for i in ema.argsort()[::-1][:3])

    if arr.std() < 8:                     # cuadro vacío / cámara tapada
        txt = "<b style='color:#aaa'>Sin imagen: muestra un platillo a la cámara</b>"
    elif conf < d["thr"]:                 # comportamiento de baja confianza: se abstiene
        txt = f"<b style='color:#ffb74d'>No estoy seguro</b> ({conf:.0%} &lt; umbral {d['thr']:.0%})"
    elif clase == OTHER_CLASS:
        txt = f"<b style='color:#90caf9'>Otro platillo (no mexicano)</b> {conf:.0%}"
    else:
        txt = (f"<b style='color:#81c784;font-size:22px'>{DISPLAY[clase]} {conf:.0%}</b><br>"
               f"~{KCAL[clase]} kcal por porción (aprox.)")
    etiqueta = f"{txt}<br><span style='font-size:13px;opacity:.85'>Top-3: {top3} | {fps:.1f} FPS | modelo {lat_ms[-1]:.0f} ms</span>"

if lat_ms:
    print(f"Cuadros: {len(lat_ms)} | latencia media del modelo: {np.mean(lat_ms):.1f} ms | FPS finales: {fps:.1f}")


### 11b. Alternativa: una sola foto  <!-- WEBCAM_NUTRIVISION -->
Si tu internet es lento y el video en vivo se traba, usa esta celda: toma **una foto** y la clasifica.

In [ ]:
# WEBCAM_NUTRIVISION: una sola foto
JS_FOTO = """
async function nvFoto(){
  const div=document.createElement('div'); const btn=document.createElement('button'); btn.textContent='Tomar foto';
  const v=document.createElement('video'); v.style.cssText='width:480px;display:block'; v.playsInline=true;
  div.appendChild(v); div.appendChild(btn); document.body.appendChild(div);
  const st=await navigator.mediaDevices.getUserMedia({video:true}); v.srcObject=st; await v.play();
  google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
  await new Promise(r=>btn.onclick=r);
  const c=document.createElement('canvas'); c.width=v.videoWidth; c.height=v.videoHeight;
  c.getContext('2d').drawImage(v,0,0); st.getVideoTracks()[0].stop(); div.remove();
  return c.toDataURL('image/jpeg',0.9);
}
"""
display(Javascript(JS_FOTO))
foto = Image.open(io.BytesIO(base64.b64decode(eval_js("nvFoto()").split(",", 1)[1])))
p = clasificar(foto)
import matplotlib.pyplot as plt
orden = p.argsort()[::-1]
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(foto); ax[0].axis("off")
k = orden[0]; c = CLASSES[k]
titulo = "No estoy seguro" if p[k] < 0.55 else (DISPLAY[c] + (f"  (~{KCAL[c]} kcal)" if KCAL[c] else ""))
ax[0].set_title(f"{titulo}  {p[k]:.0%}")
ax[1].barh([DISPLAY[CLASSES[i]] for i in orden[:5]][::-1], p[orden[:5]][::-1]); ax[1].set_xlim(0, 1)
ax[1].set_title("Top-5 probabilidades"); plt.tight_layout(); plt.show()
